# **Model Training**

In [10]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBRegressor

In [11]:
file_path = Path.cwd().parent.joinpath("data/cleaned_demand_forecasting.parquet")
df = pd.read_parquet(file_path)
df.head(1)

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,...,Competitor Pricing,Seasonality,Epidemic,Demand,Year,Weekday,Day,Month,DiscountedPrice,SellThroughRate
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,...,85.73,Winter,0,115,2022,Saturday,1,1,69.084,0.523077


## Model training and evaluation function

In [12]:
def model_trainer_and_evaluator(X, y) -> None:
    algos = [
        LinearRegression(),
        Ridge(random_state=42),
        Lasso(random_state=42),
        ElasticNet(random_state=42),
        XGBRegressor(),
    ]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    X_train, X_test = X_train.copy(), X_test.copy()

    def preprocessing(X_train, X_test, y_train, y_test):
        num_cols = X_train.select_dtypes(exclude="object").columns
        cat_cols = X_train.select_dtypes(include="object").columns

        for col in cat_cols:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col])
            X_test[col] = le.transform(X_test[col])

        for col in num_cols:
            std_train = StandardScaler()
            X_train[col] = std_train.fit_transform(X_train[[col]])
            X_test[col] = std_train.transform(X_test[[col]])

        std_target = StandardScaler()
        y_train_scaled = std_target.fit_transform(
            y_train.values.reshape(-1, 1)
        ).flatten()
        y_test_scaled = std_target.transform(y_test.values.reshape(-1, 1)).flatten()

        return X_train, X_test, y_train_scaled, y_test_scaled

    X_train, X_test, y_train, y_test = preprocessing(
        X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test
    )

    def model_trainer(X_train, y_train):
        models: list[object] = []
        for algo in algos:
            models.append(algo.fit(X_train, y_train))
        return models

    def model_evaluator(models, X_test, y_test) -> None:
        for model in models:
            model_name = type(model).__name__
            y_pred = model.predict(X_test)

            print(f"\n=== {model_name} ===")
            print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
            print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
            print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

    models: list = model_trainer(X_train=X_train, y_train=y_train)
    model_evaluator(models=models, X_test=X_test, y_test=y_test)


## Feature selection for training

In [13]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount',
       'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality',
       'Epidemic', 'Demand', 'Year', 'Weekday', 'Day', 'Month',
       'DiscountedPrice', 'SellThroughRate'],
      dtype='str')

### All feature

In [14]:
X = df.drop(columns=["Demand", "Date"])
y = df["Demand"]
model_trainer_and_evaluator(X=X, y=y)


=== LinearRegression ===
MAE: 0.3555
MSE: 0.2288
R2 Score: 0.7715

=== Ridge ===
MAE: 0.3555
MSE: 0.2288
R2 Score: 0.7715

=== Lasso ===
MAE: 0.7888
MSE: 1.0016
R2 Score: -0.0000

=== ElasticNet ===
MAE: 0.6460
MSE: 0.6791
R2 Score: 0.3220

=== XGBRegressor ===
MAE: 0.2373
MSE: 0.0987
R2 Score: 0.9015


In [15]:
X.shape

(76000, 20)

> The `XGBRegressor` is significantly better than others

In [16]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Category', 'Region',
       'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount',
       'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality',
       'Epidemic', 'Demand', 'Year', 'Weekday', 'Day', 'Month',
       'DiscountedPrice', 'SellThroughRate'],
      dtype='str')

In [21]:
df.groupby("Store ID")["Units Sold"].sum()

Store ID
S001    1318134
S002    1386126
S003    1375613
S004    1308191
S005    1362812
Name: Units Sold, dtype: int64

In [24]:
df.groupby(["Product ID", "Store ID"])["Units Sold"].sum()


Product ID  Store ID
P0001       S001        62075
            S002        80654
            S003        58428
            S004        80941
            S005        74716
                        ...  
P0020       S001        49957
            S002        48824
            S003        73031
            S004        47817
            S005        58332
Name: Units Sold, Length: 100, dtype: int64

In [ ]:
df["ProductStoreRatio"] = df.groupby(["Product ID", "Store ID"])["Units Sold"].sum()
df["ProductStoreRatio"]

In [17]:
df["Product ID"].value_counts()

Product ID
P0001    3800
P0002    3800
P0003    3800
P0004    3800
P0005    3800
P0006    3800
P0007    3800
P0008    3800
P0009    3800
P0010    3800
P0011    3800
P0012    3800
P0013    3800
P0014    3800
P0015    3800
P0016    3800
P0017    3800
P0018    3800
P0019    3800
P0020    3800
Name: count, dtype: int64

In [18]:
df["Store ID"].value_counts()

Store ID
S001    15200
S002    15200
S003    15200
S004    15200
S005    15200
Name: count, dtype: int64

### Drop high cardinality features

In [9]:
X = df.drop(columns=["Demand", "Date", "Store ID", "Product ID"])
y = df["Demand"]
model_trainer_and_evaluator(X=X, y=y)


=== LinearRegression ===
MAE: 0.3570
MSE: 0.2305
R2 Score: 0.7698

=== Ridge ===
MAE: 0.3570
MSE: 0.2305
R2 Score: 0.7698

=== Lasso ===
MAE: 0.7888
MSE: 1.0016
R2 Score: -0.0000

=== ElasticNet ===
MAE: 0.6467
MSE: 0.6804
R2 Score: 0.3207

=== XGBRegressor ===
MAE: 0.2620
MSE: 0.1207
R2 Score: 0.8795


### Drop redundant features

In [7]:
X = df.drop(columns=["Demand", "Date", "Store ID", "Product ID", "Weekday"])
y = df["Demand"]
model_trainer_and_evaluator(X=X, y=y)



=== LinearRegression ===
MAE: 0.3570
MSE: 0.2305
R2 Score: 0.7698

=== Ridge ===
MAE: 0.3570
MSE: 0.2305
R2 Score: 0.7698

=== Lasso ===
MAE: 0.7888
MSE: 1.0016
R2 Score: -0.0000

=== ElasticNet ===
MAE: 0.6467
MSE: 0.6804
R2 Score: 0.3207

=== XGBRegressor ===
MAE: 0.2627
MSE: 0.1216
R2 Score: 0.8786


### Drop low correlated features(<10%)

In [8]:
X = df.drop(
    columns=["Demand", "Date", "Price", "Day", "Month", "Year", "DiscountedPrice"]
)
y = df["Demand"]
model_trainer_and_evaluator(X=X, y=y)


=== LinearRegression ===
MAE: 0.3555
MSE: 0.2288
R2 Score: 0.7716

=== Ridge ===
MAE: 0.3555
MSE: 0.2288
R2 Score: 0.7716

=== Lasso ===
MAE: 0.7888
MSE: 1.0016
R2 Score: -0.0000

=== ElasticNet ===
MAE: 0.6460
MSE: 0.6791
R2 Score: 0.3220

=== XGBRegressor ===
MAE: 0.2549
MSE: 0.1113
R2 Score: 0.8889


### Combine both 

In [9]:
X = df.drop(
    columns=[
        "Demand",
        "Date",
        "Store ID",
        "Product ID",
        "Weekday",
        "Demand",
        "Date",
        "Price",
        "Day",
        "Month",
        "Year",
        "DiscountedPrice",
    ]
)
y = df["Demand"]
model_trainer_and_evaluator(X=X, y=y)



=== LinearRegression ===
MAE: 0.3570
MSE: 0.2305
R2 Score: 0.7699

=== Ridge ===
MAE: 0.3570
MSE: 0.2305
R2 Score: 0.7699

=== Lasso ===
MAE: 0.7888
MSE: 1.0016
R2 Score: -0.0000

=== ElasticNet ===
MAE: 0.6467
MSE: 0.6804
R2 Score: 0.3207

=== XGBRegressor ===
MAE: 0.2681
MSE: 0.1255
R2 Score: 0.8747


### Hyperparameter tunining

In [10]:
X = df.drop(columns=["Demand", "Date"])
y = df["Demand"]
model_trainer_and_evaluator(X=X, y=y)


=== LinearRegression ===
MAE: 0.3555
MSE: 0.2288
R2 Score: 0.7715

=== Ridge ===
MAE: 0.3555
MSE: 0.2288
R2 Score: 0.7715

=== Lasso ===
MAE: 0.7888
MSE: 1.0016
R2 Score: -0.0000

=== ElasticNet ===
MAE: 0.6460
MSE: 0.6791
R2 Score: 0.3220

=== XGBRegressor ===
MAE: 0.2373
MSE: 0.0987
R2 Score: 0.9015


In [13]:
def preprocessing(X_train, X_test, y_train, y_test):
    num_cols = X_train.select_dtypes(exclude="object").columns
    cat_cols = X_train.select_dtypes(include="object").columns

    for col in cat_cols:
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col])
        X_test[col] = le.transform(X_test[col])

    for col in num_cols:
        std_train = StandardScaler()
        X_train[col] = std_train.fit_transform(X_train[[col]])
        X_test[col] = std_train.transform(X_test[[col]])

    std_target = StandardScaler()
    y_train_scaled = std_target.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_test_scaled = std_target.transform(y_test.values.reshape(-1, 1)).flatten()

    return X_train, X_test, y_train_scaled, y_test_scaled


In [14]:
param_grid = {
    "n_estimators": [300, 500, 800],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "reg_alpha": [0, 0.1, 0.5],
    "reg_lambda": [1, 1.5, 2],
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_test, y_train, y_test = preprocessing(X_train, X_test, y_train, y_test)

xgb_model = XGBRegressor(random_state=42)
model = RandomizedSearchCV(
    xgb_model, param_grid, n_iter=50, cv=5, scoring="r2", verbose=4, n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n=== xgboost ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"MSE: {mean_squared_error(y_test, y_pred):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")


Fitting 5 folds for each of 50 candidates, totalling 250 fits

=== xgboost ===
MAE: 0.2635
MSE: 0.1210
R2 Score: 0.8792
